<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/propensity_metadata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import packages

In [220]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

In [253]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import CSV Files

In [221]:
men_meta = pd.read_csv('/content/Mens_metadata_12_3.csv')
men_meta = pd.read_csv('/content/Mens_metadata_12_3.csv')
women_meta_crime = pd.read_csv('/content/women_meta_crime.csv')
save_path = '/content/drive/MyDrive/Capital_trials/'
full_meta = pd.read_csv(save_path + 'full_crime_meta_.csv')


#Reformat Metadata to Enable Propensity Score Matching

In [222]:
lst = men_meta['Aggrevating factors/special circumstace'].unique()
mens_factors = [str(x) for x in lst]

In [223]:
women_meta_crime.Name.unique()

array(['Angelina Rodriguez', 'Blanche Taylor Moore', 'Brenda E. Andrew',
       'Donna Marie Roberts', 'Lorraine Alison Hunter', 'Wendi Andriano',
       'Valerie Martin', 'Robin Lee Row', 'Manling Williams',
       'Belinda Magaña', 'Tierra Capri Gobble', 'Tiffany Moss',
       'Susan Eubanks', 'Patricia Blackmon', 'Christie Michelle Scott',
       'Lisa Carpenter Graham', 'Cathy Sarinana', 'Michelle Sue Tharp',
       'Veronica Gonzales', 'Heather Leavell Keaton', 'Melissa Lucio',
       'Maria Alfaro', 'Socorro Caro', 'Sammantha Allen',
       'Darlie Routier', 'Taylor Parker\n', 'Michelle Michaud',
       'Cherie Rhoades', 'Brittany Holberg', 'Brooke Rottiers',
       'Carlette Parker', 'Tiffany Cole', 'Christa Pike', 'Tina Brown',
       'Kerry Dalton', 'Virginia Caudill', 'Maureen McDermott',
       'Kimberly Cargill', 'Linda Carty', 'Margaret Allen',
       'Tanya Nelson', 'Antoinette Frank', 'Janeen Snyder',
       'Celeste Carrington', 'Cynthia Coffman', 'Erica Sheppard',
    

In [224]:
#clean errors and save a list of unique factors
women_meta_crime.replace({'Name': {'Taylor Parker\n': 'Taylor Parker'}}, inplace=True)
lst = women_meta_crime.aggravating_factors_list.unique()
womens_factors = [str(x) for x in lst]

In [225]:
#function to seperate aggrevating factors (crimes) in a list
def seperate_agg_factors(factors_list):
  processed_list = []
  for item in factors_list:
      if ',' in item:
          processed_list.extend(item.split(','))
      else:
          processed_list.append(item)
  return processed_list

In [226]:
#get list of aggrevating factors and strip ending and trailing whitespace, then get unique factors
format_women_factors = seperate_agg_factors(womens_factors)
format_men_factors = seperate_agg_factors(mens_factors)

format_men_factors = [item.lstrip() for item in format_men_factors]
format_women_factors = [item.lstrip() for item in format_women_factors]

format_men_factors = [item.rstrip() for item in format_men_factors]
format_women_factors = [item.rstrip() for item in format_women_factors]

unique_women_factors = list(set(format_women_factors))
unique_men_factors = list(set(format_men_factors))

In [227]:
#create a mapping of how women's factors map to mens factors, men's factors to update,

mapping = {
    #replace from womens set
  'Motive - financial gain': 'financial gain',
  'Motive - interfere with prosecution/arrest' : 'interfere with prosecution/arrest',

  'Weapon - poison': 'poison',
  'Weapon - gun': 'personal use of a handgun',

  'Manner - HAC': 'especially cruel',
  'Manner - torture' :  'torture',

  'Crime - robbery/burglary' : 'burglary/robbery',
  'Crime - kidnapping' : 'kidnapping',
  'Crime - rape' : 'rape',

  'Victim - child' : 'child victim',
  'Victim - officer' : 'murdered an officer in the line of duty',
  'Multiple - multiple murders' : 'multiple murders',

  'Premeditated' : 'deliberate and premeditated',

  'Previous serious offense': 'prior offense',

#replace from mens set
  'killing to prevent witness testifying': 'interfere with prosecution/arrest',
  'committed the murder to avoid arrest': 'interfere with prosecution/arrest',
  'rape/attempted rape': 'rape',
  'knowingly created a great risk\nof death to more than one person in a public place' :'created public risk',
  'unlawful possessio of a firearm' : 'unlawful possession of a firearm',
  'multiple murder' :'multiple murders',
}

#options in women not represented in mens set:
#'Continuing threat to society', 'Disregard for human life',
# Weapon - deadly weapon,  'Manner - mayhem', 'Weapon - arson',

#options in men not represented in womens set:
#lying in wait,  lewd acts with a child,
# sodomy, gang murder,
#larceny, attempted rape, sexual assult, created public risk
#shooting at an occupied vehicle, 'prior conviction',

In [228]:
men = men_meta[['Name', 'Aggrevating factors/special circumstace']]
women = women_meta_crime[['Name', 'aggravating_factors_list']]

In [229]:
men = men.rename(columns={'Aggrevating factors/special circumstace': 'aggravating_factors_list'})

In [230]:
all_gender_crime = pd.concat([men, women])

In [231]:
all_gender_crime['aggravating_factors_listed'] = all_gender_crime['aggravating_factors_list'].str.split(',')
all_gender_crime

,Name,aggravating_factors_list,aggravating_factors_listed
0,"Dearman, Derrick",multiple murders,[multiple murders]
1,"DeBlase, John Joseph","multiple murders, child victim","[multiple murders, child victim]"
2,"Kennedy, Carlos","burglary/robbery, rape","[burglary/robbery, rape]"
3,"Lackey, Andrew",burglary/robbery,[burglary/robbery]
4,"Mills, Jamie","multiple murders, burglary/robbery","[multiple murders, burglary/robbery]"
...,...,...,...
43,Celeste Carrington,"Multiple - multiple murders, Crime - robbery/b...","[Multiple - multiple murders, Crime - robbery..."
44,Cynthia Coffman,"Weapon - gun, Crime - robbery/burglary","[Weapon - gun, Crime - robbery/burglary]"
45,Erica Sheppard,Crime - robbery/burglary,[Crime - robbery/burglary]
46,Shawna Forde,"Previous serious offense, Motive - financial g...","[Previous serious offense, Motive - financial..."


In [232]:
all_gender_crime["aggravating_factors_listed"] = all_gender_crime["aggravating_factors_listed"].apply(
    lambda x: [s.strip() for s in x] if isinstance(x, list) else x
)

In [233]:
all_gender_crime["aggravating_factors_listed"] = all_gender_crime["aggravating_factors_listed"].apply(
    lambda x: [mapping.get(item, item) for item in x] if isinstance(x, list) else x
)

In [234]:
all_gender_crime

,Name,aggravating_factors_list,aggravating_factors_listed
0,"Dearman, Derrick",multiple murders,[multiple murders]
1,"DeBlase, John Joseph","multiple murders, child victim","[multiple murders, child victim]"
2,"Kennedy, Carlos","burglary/robbery, rape","[burglary/robbery, rape]"
3,"Lackey, Andrew",burglary/robbery,[burglary/robbery]
4,"Mills, Jamie","multiple murders, burglary/robbery","[multiple murders, burglary/robbery]"
...,...,...,...
43,Celeste Carrington,"Multiple - multiple murders, Crime - robbery/b...","[multiple murders, burglary/robbery]"
44,Cynthia Coffman,"Weapon - gun, Crime - robbery/burglary","[personal use of a handgun, burglary/robbery]"
45,Erica Sheppard,Crime - robbery/burglary,[burglary/robbery]
46,Shawna Forde,"Previous serious offense, Motive - financial g...","[prior offense, financial gain, multiple murde..."


In [70]:
all_gender_crime.to_csv('normalized_crime_metadata.csv')

In [235]:
dummies = all_gender_crime.aggravating_factors_listed.explode().str.get_dummies().groupby(level=0).max()


In [236]:
dummies.columns

Index(['Continuing threat to society', 'Disregard for human life',
       'Manner - mayhem', 'Multiple - harm to multiple people', 'Retaliation',
       'Victim - old or disabled', 'Weapon - arson', 'Weapon - deadly weapon',
       'aggravated assault', 'attempted rape', 'burglary/robbery',
       'child victim', 'created public risk', 'deliberate and premeditated',
       'done while committing multiplie felonies', 'especially cruel',
       'fiancial gain', 'financial gain', 'financial gian', 'gang murder',
       'intentional infliction of great bodily injury',
       'interfere with prosecution/arrest', 'kidnapping', 'larceny',
       'lewd acts with a child', 'lying in wait', 'multiple murders',
       'murdered an officer in the line of duty', 'personal use of a handgun',
       'poison', 'prior conviction', 'prior offense', 'prior offenses', 'rape',
       'robbery', 'sexual assult', 'shooting at an occupied vehicle', 'sodomy',
       'torture', 'unlawful possession of a firearm

In [237]:
dummies = (
    all_gender_crime["aggravating_factors_listed"]
    .apply(lambda x: x if isinstance(x, list) else [])
    .explode()
    .str.get_dummies()
    .groupby(level=0).max()   # critical: restore unique row index
)

In [238]:
name_col = all_gender_crime["Name"].reset_index(drop=True)
dummies = dummies.reset_index(drop=True)

meta_encoded = pd.concat([name_col, dummies], axis=1)


In [244]:
crime_meta_encoded = pd.concat([all_gender_crime['Name'], dummies], axis=1)


In [85]:
crime_meta_encoded.to_csv('ohe_crime_metadata.csv')

In [246]:
crime_meta_encoded

,Name,Continuing threat to society,Disregard for human life,Manner - mayhem,Multiple - harm to multiple people,Retaliation,Victim - old or disabled,Weapon - arson,Weapon - deadly weapon,aggravated assault,...,prior conviction,prior offense,prior offenses,rape,robbery,sexual assult,shooting at an occupied vehicle,sodomy,torture,unlawful possession of a firearm
0,"Dearman, Derrick",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,"DeBlase, John Joseph",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,"Kennedy, Carlos",1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,"Lackey, Andrew",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,"Mills, Jamie",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43,Celeste Carrington,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
44,Cynthia Coffman,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
45,Erica Sheppard,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
46,Shawna Forde,0,0,0,0,0,0,0,0,0,...,0,1,0,1,0,0,0,0,0,0


In [243]:
full_meta.replace({'Name': {'Carlette Parker ': 'Carlette Parker'}}, inplace=True)
full_meta


,Unnamed: 0,document_name,Name,State,Race/Ethnicity,gender
0,0,Christie Michelle Scott.txt,Christie Michelle Scott,AL,White,women
1,1,Heather Keaton.txt,Heather Leavell Keaton,AL,"Native American,White,Black",women
2,2,Lisa Graham.txt,Lisa Carpenter Graham,AL,White,women
3,3,Patricia Blackmon.txt,Patricia Blackmon,AL,Black,women
4,4,Tierra Capri Gobble.txt,Tierra Capri Gobble,AL,White,women
...,...,...,...,...,...,...
100,100,Manuel Velez Combined Transcript.txt,"Velez, Manuel",TX,Latino,men
101,101,Willie Terion Washington Combined Transcript.txt,"Washington, Willie Terion",TX,Black,men
102,102,Perry Williams Combined Transcript.txt,"Williams, Perry E",TX,Black,men
103,103,David Leonard Wood Combined Transcript.txt,"Wood, David Leonard",TX,White,men


/tmp/ipykernel_284/2603687059.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  full_meta['Name'] = full_meta['Name'].replace({'Taylor Parker ': 'Taylor Parker'}, inplace=True)


In [247]:
merged_meta_ohe = pd.merge(full_meta, crime_meta_encoded, on='Name', indicator=True)


In [248]:
merged_meta_ohe

,Unnamed: 0,document_name,Name,State,Race/Ethnicity,gender,Continuing threat to society,Disregard for human life,Manner - mayhem,Multiple - harm to multiple people,...,prior offense,prior offenses,rape,robbery,sexual assult,shooting at an occupied vehicle,sodomy,torture,unlawful possession of a firearm,_merge
0,0,Christie Michelle Scott.txt,Christie Michelle Scott,AL,White,women,0,0,0,0,...,0,0,0,0,0,0,0,0,0,both
1,1,Heather Keaton.txt,Heather Leavell Keaton,AL,"Native American,White,Black",women,0,0,0,0,...,0,0,0,0,0,0,0,0,0,both
2,2,Lisa Graham.txt,Lisa Carpenter Graham,AL,White,women,0,0,0,0,...,0,0,0,0,0,1,0,0,1,both
3,3,Patricia Blackmon.txt,Patricia Blackmon,AL,Black,women,0,0,0,0,...,0,0,0,0,0,0,0,0,0,both
4,4,Tierra Capri Gobble.txt,Tierra Capri Gobble,AL,White,women,0,0,0,0,...,0,0,0,0,0,0,0,0,0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,100,Manuel Velez Combined Transcript.txt,"Velez, Manuel",TX,Latino,men,0,0,0,0,...,0,0,0,0,0,0,0,0,0,both
101,101,Willie Terion Washington Combined Transcript.txt,"Washington, Willie Terion",TX,Black,men,0,0,0,0,...,0,0,0,0,0,0,0,0,0,both
102,102,Perry Williams Combined Transcript.txt,"Williams, Perry E",TX,Black,men,0,0,0,0,...,0,0,0,0,0,0,0,0,0,both
103,103,David Leonard Wood Combined Transcript.txt,"Wood, David Leonard",TX,White,men,0,0,0,0,...,0,0,0,0,0,0,0,0,0,both


In [249]:
full_meta[full_meta['gender']=='women'].Name.unique()

array(['Christie Michelle Scott', 'Heather Leavell Keaton',
       'Lisa Carpenter Graham', 'Patricia Blackmon',
       'Tierra Capri Gobble', 'Sammantha Allen', 'Shawna Forde',
       'Wendi Andriano', 'Angelina Rodriguez', 'Brooke Rottiers',
       'Celeste Carrington', 'Cherie Rhoades', 'Kerry Dalton',
       'Lorraine Alison Hunter', 'Manling Williams', 'Maria Alfaro',
       'Maureen McDermott', 'Socorro Caro', 'Susan Eubanks',
       'Tanya Nelson', 'Valerie Martin', 'Veronica Gonzales',
       'Margaret Allen', 'Tiffany Cole', 'Tina Brown', 'Tiffany Moss',
       'Antoinette Frank', 'Lisa Jo Chamberlin', 'Blanche Taylor Moore',
       'Carlette Parker', 'Donna Marie Roberts', 'Brenda E. Andrew',
       'Michelle Sue Tharp', 'Christa Pike', 'Brittany Holberg',
       'Darlie Routier', 'Erica Sheppard', 'Kimberly Cargill',
       'Melissa Lucio', 'Taylor Parker'], dtype=object)

In [250]:
full_meta.gender.value_counts()

,count
gender,
men,65
women,40


In [251]:
merged_meta_ohe.gender.value_counts()

,count
gender,
men,65
women,40


In [252]:
#sanity check that the merge worked and no defendants are lost (set should be empty)
print(set(full_meta[full_meta['gender']=='women'].Name.unique()) - set(merged_meta_ohe[merged_meta_ohe['gender']=='women'].Name.unique()) )
print(set(full_meta[full_meta['gender']=='men'].Name.unique()) - set(merged_meta_ohe[merged_meta_ohe['gender']=='men'].Name.unique()) )

set()
set()


In [159]:
merged_meta_ohe.to_csv('ohe_crime_metadata_.csv')

In [160]:
merged_meta_ohe.columns

Index(['Unnamed: 0', 'document_name', 'Name', 'State', 'Race/Ethnicity',
       'gender', 'state_race_propensity', 'Continuing threat to society',
       'Disregard for human life', 'Manner - mayhem',
       'Multiple - harm to multiple people', 'Retaliation',
       'Victim - old or disabled', 'Weapon - arson', 'Weapon - deadly weapon',
       'aggravated assault', 'attempted rape', 'burglary/robbery',
       'child victim', 'created public risk', 'deliberate and premeditated',
       'done while committing multiplie felonies', 'especially cruel',
       'fiancial gain', 'financial gain', 'financial gian', 'gang murder',
       'intentional infliction of great bodily injury',
       'interfere with prosecution/arrest', 'kidnapping', 'larceny',
       'lewd acts with a child', 'lying in wait', 'multiple murders',
       'murdered an officer in the line of duty', 'personal use of a handgun',
       'poison', 'prior conviction', 'prior offense', 'prior offenses', 'rape',
       'robbery'

In [161]:
df_encoded = pd.get_dummies(merged_meta_ohe, columns=['State', 'Race/Ethnicity', 'gender'], dtype=str)
# print(df_encoded.columns.tolist())
cols_to_convert = ['State_AL', 'State_AZ', 'State_CA','State_FL', 'State_GA',
 'State_LA', 'State_MS', 'State_NC', 'State_OH',
 'State_OK', 'State_PA', 'State_SC', 'State_TN', 'State_TX',
  'Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White', 'gender_women', 'gender_men']
df_encoded = df_encoded.replace('', '0')

df_encoded[cols_to_convert] = df_encoded[cols_to_convert].astype(float)
print(df_encoded['gender_women'].unique())

print(df_encoded.columns.tolist())


[1. 0.]
['Unnamed: 0', 'document_name', 'Name', 'state_race_propensity', 'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem', 'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled', 'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape', 'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated', 'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain', 'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest', 'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders', 'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction', 'prior offense', 'prior offenses', 'rape', 'robbery', 'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm', '_merge', 'State_AL', 

#Calculate and Save Propensity Scores

In [163]:
#race crime and state
X = df_encoded[['State_AL', 'State_AZ', 'State_CA','State_FL', 'State_GA',
 'State_LA', 'State_MS', 'State_NC', 'State_OH',
 'State_OK', 'State_PA', 'State_SC', 'State_TN', 'State_TX',
  'Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm',]]
T = df_encoded['gender_women']


model = LogisticRegression(random_state=0)
# model.fit(X, T)

X = X.replace(r'^\s*$', pd.NA, regex=True)  # '' -> NA
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(0)

T = pd.to_numeric(T, errors="coerce")
mask = T.notna()
X, T = X.loc[mask], T.loc[mask]

model.fit(X, T.astype(int))


crime_race_state_propensity = model.predict_proba(X)[:, 1]



full_meta['race_crime_state_prop'] = crime_race_state_propensity


#race and crime
X = df_encoded[['Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm',]]
T = df_encoded['gender_women']


model = LogisticRegression(random_state=0)
# model.fit(X, T)

X = X.replace(r'^\s*$', pd.NA, regex=True)  # '' -> NA
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(0)

T = pd.to_numeric(T, errors="coerce")
mask = T.notna()
X, T = X.loc[mask], T.loc[mask]

model.fit(X, T.astype(int))


crime_race_propensity = model.predict_proba(X)[:, 1]

full_meta['race_crime_prop'] = crime_race_propensity

full_meta


,Unnamed: 0,document_name,Name,State,Race/Ethnicity,gender,state_race_propensity,race_crime_state_prop,race_crime_prop
0,0,Christie Michelle Scott.txt,Christie Michelle Scott,AL,White,women,0.476238,0.600540,0.636801
1,1,Heather Keaton.txt,Heather Leavell Keaton,AL,"Native American,White,Black",women,0.603035,0.685484,0.715810
2,2,Lisa Graham.txt,Lisa Carpenter Graham,AL,White,women,0.476238,0.443808,0.461215
3,3,Patricia Blackmon.txt,Patricia Blackmon,AL,Black,women,0.373644,0.333647,0.380188
4,4,Tierra Capri Gobble.txt,Tierra Capri Gobble,AL,White,women,0.476238,0.444600,0.497139
...,...,...,...,...,...,...,...,...,...
100,106,Manuel Velez Combined Transcript.txt,"Velez, Manuel",TX,Latino,men,0.110893,0.057492,0.057825
101,107,Willie Terion Washington Combined Transcript.txt,"Washington, Willie Terion",TX,Black,men,0.214550,0.173717,0.191333
102,108,Perry Williams Combined Transcript.txt,"Williams, Perry E",TX,Black,men,0.214550,0.173717,0.191333
103,109,David Leonard Wood Combined Transcript.txt,"Wood, David Leonard",TX,White,men,0.293962,0.251051,0.274495


In [165]:
full_meta.to_csv('metadata_matched_propensity_scores.csv')

In [177]:
df_encoded['race_crime_prop'] = crime_race_propensity

df_encoded['race_crime_state_prop'] = crime_race_state_propensity


In [210]:
df_encoded.to_csv(save_path + 'ohe_metadata_with_prop.csv')

NameError: name 'save_path' is not defined

In [179]:
df_encoded

,Unnamed: 0,document_name,Name,state_race_propensity,Continuing threat to society,Disregard for human life,Manner - mayhem,Multiple - harm to multiple people,Retaliation,Victim - old or disabled,...,Race/Ethnicity_Black,Race/Ethnicity_Latino,Race/Ethnicity_Latinx,Race/Ethnicity_Native American,"Race/Ethnicity_Native American,White,Black",Race/Ethnicity_White,gender_men,gender_women,race_crime_prop,race_crime_state_prop
0,0,Christie Michelle Scott.txt,Christie Michelle Scott,0.476238,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.636801,0.600540
1,1,Heather Keaton.txt,Heather Leavell Keaton,0.603035,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.715810,0.685484
2,2,Lisa Graham.txt,Lisa Carpenter Graham,0.476238,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.461215,0.443808
3,3,Patricia Blackmon.txt,Patricia Blackmon,0.373644,0,0,0,0,0,0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.380188,0.333647
4,4,Tierra Capri Gobble.txt,Tierra Capri Gobble,0.476238,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.497139,0.444600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,106,Manuel Velez Combined Transcript.txt,"Velez, Manuel",0.110893,0,0,0,0,0,0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.057825,0.057492
101,107,Willie Terion Washington Combined Transcript.txt,"Washington, Willie Terion",0.214550,0,0,0,0,0,0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.191333,0.173717
102,108,Perry Williams Combined Transcript.txt,"Williams, Perry E",0.214550,0,0,0,0,0,0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.191333,0.173717
103,109,David Leonard Wood Combined Transcript.txt,"Wood, David Leonard",0.293962,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.274495,0.251051


#Create New matched Groups based on Scores

taking a caliper + NN approach
[link text](https://)

In [180]:
#first get caliper

df_encoded['race_crime_state_prop_logit'] = np.log(df_encoded['race_crime_state_prop'] / (1 - df_encoded['race_crime_state_prop']))

# 2. Define Caliper (0.2 * SD of logit_ps)
caliper_state_race_crime = 0.2 * df_encoded['race_crime_state_prop_logit'].std()


df_encoded['race_crime_prop_logit'] = np.log(df_encoded['race_crime_prop'] / (1 - df_encoded['race_crime_prop']))

# 2. Define Caliper (0.2 * SD of logit_ps)
caliper_race_crime = 0.2 * df_encoded['race_crime_prop_logit'].std()

In [181]:
print(caliper_race_crime)
print(caliper_state_race_crime)

0.21972522550981732
0.2299348551729465


In [196]:
#based on geeks for geeks propensty score implementation

#match groups based on propensity, start with race, crime, state group

women_group = df_encoded[df_encoded['gender_women'] == 1].reset_index().rename(columns={"index": "women_idx"})
men_group = df_encoded[df_encoded['gender_women'] == 0].reset_index().rename(columns={"index": "men_idx"})

nn = NearestNeighbors(n_neighbors=1)
nn.fit(men_group[['race_crime_state_prop']])
distances, indices = nn.kneighbors(women_group[['race_crime_state_prop']])

pairs = []
seen_men = set()

for i, (dist, j) in enumerate(zip(distances.ravel(), indices.ravel())):
    m_idx = int(men_group.loc[j, "men_idx"])
    if m_idx in seen_men:  # without replacement
        continue
    if dist > caliper_state_race_crime:
        continue

    seen_men.add(m_idx)
    pairs.append({
        "women_idx": int(women_group.loc[i, "women_idx"]),
        "men_idx": m_idx,
        "women_prop": float(women_group.loc[i, 'race_crime_state_prop']),
        "men_prop": float(men_group.loc[j, 'race_crime_state_prop']),
        "distance": float(dist),
    })

final = pd.DataFrame(pairs)

In [197]:
final

,women_idx,men_idx,women_prop,men_prop,distance
0,0,56,0.600540,0.576568,0.023972
1,1,41,0.685484,0.662235,0.023250
2,2,47,0.443808,0.451718,0.007911
3,3,58,0.333647,0.343969,0.010321
4,5,40,0.392700,0.401060,0.008360
5,6,54,0.541121,0.534228,0.006894
6,8,46,0.784870,0.743937,0.040933
7,9,59,0.486777,0.462144,0.024633
8,10,65,0.271519,0.271519,0.000000
9,11,64,0.426702,0.426670,0.000033


In [198]:
matched_rows = df_encoded.loc[final["women_idx"].tolist() + final["men_idx"].tolist()]


In [199]:
race_crime_state_prop_df = matched_rows

In [200]:
#based on geeks for geeks propensty score implementation

#match groups based on propensity, start with race, crime, state group

women_group = df_encoded[c['gender_women'] == 1].reset_index().rename(columns={"index": "women_idx"})
men_group = df_encoded[df_encoded['gender_women'] == 0].reset_index().rename(columns={"index": "men_idx"})

nn = NearestNeighbors(n_neighbors=1)
nn.fit(men_group[['race_crime_prop']])
distances, indices = nn.kneighbors(women_group[['race_crime_prop']])

pairs = []
seen_men = set()

for i, (dist, j) in enumerate(zip(distances.ravel(), indices.ravel())):
    m_idx = int(men_group.loc[j, "men_idx"])
    if m_idx in seen_men:  # without replacement
        continue
    if dist > caliper_state_race_crime:
        continue

    seen_men.add(m_idx)
    pairs.append({
        "women_idx": int(women_group.loc[i, "women_idx"]),
        "men_idx": m_idx,
        "women_prop": float(women_group.loc[i, 'race_crime_prop']),
        "men_prop": float(men_group.loc[j, 'race_crime_prop']),
        "distance": float(dist),
    })

final_race_crime = pd.DataFrame(pairs)

In [201]:
matched_rows2 = df_encoded.loc[final["women_idx"].tolist() + final["men_idx"].tolist()]
race_crime_prop_df = matched_rows2

In [204]:
state_race_crime_names = race_crime_state_prop_df.Name.to_list()
race_crime_names = race_crime_prop_df.Name.to_list()

In [207]:
mask = df_encoded['Name'].isin(race_crime_names)
df_encoded['race_crime_propensity_match'] = mask

mask = df_encoded['Name'].isin(state_race_crime_names)
df_encoded['state_race_crime_propensity_match'] = mask


In [208]:
df_encoded

,Unnamed: 0,document_name,Name,state_race_propensity,Continuing threat to society,Disregard for human life,Manner - mayhem,Multiple - harm to multiple people,Retaliation,Victim - old or disabled,...,"Race/Ethnicity_Native American,White,Black",Race/Ethnicity_White,gender_men,gender_women,race_crime_prop,race_crime_state_prop,race_crime_state_prop_logit,race_crime_prop_logit,race_crime_propensity_match,state_race_crime_propensity_match
0,0,Christie Michelle Scott.txt,Christie Michelle Scott,0.476238,0,0,0,0,0,0,...,0.0,1.0,0.0,1.0,0.636801,0.600540,0.407716,0.561505,True,True
1,1,Heather Keaton.txt,Heather Leavell Keaton,0.603035,0,0,0,0,0,0,...,1.0,0.0,0.0,1.0,0.715810,0.685484,0.779092,0.923771,True,True
2,2,Lisa Graham.txt,Lisa Carpenter Graham,0.476238,0,0,0,0,0,0,...,0.0,1.0,0.0,1.0,0.461215,0.443808,-0.225723,-0.155450,True,True
3,3,Patricia Blackmon.txt,Patricia Blackmon,0.373644,0,0,0,0,0,0,...,0.0,0.0,0.0,1.0,0.380188,0.333647,-0.691735,-0.488751,True,True
4,4,Tierra Capri Gobble.txt,Tierra Capri Gobble,0.476238,0,0,0,0,0,0,...,0.0,1.0,0.0,1.0,0.497139,0.444600,-0.222513,-0.011442,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,106,Manuel Velez Combined Transcript.txt,"Velez, Manuel",0.110893,0,0,0,0,0,0,...,0.0,0.0,1.0,0.0,0.057825,0.057492,-2.796895,-2.790768,False,False
101,107,Willie Terion Washington Combined Transcript.txt,"Washington, Willie Terion",0.214550,0,0,0,0,0,0,...,0.0,0.0,1.0,0.0,0.191333,0.173717,-1.559512,-1.441372,False,False
102,108,Perry Williams Combined Transcript.txt,"Williams, Perry E",0.214550,0,0,0,0,0,0,...,0.0,0.0,1.0,0.0,0.191333,0.173717,-1.559512,-1.441372,True,True
103,109,David Leonard Wood Combined Transcript.txt,"Wood, David Leonard",0.293962,0,0,0,0,0,0,...,0.0,1.0,1.0,0.0,0.274495,0.251051,-1.093013,-0.971937,False,False


In [209]:
df_encoded.to_csv('save_path'+'ohe_metadata_with_prop_and_matches.csv')